# Simulation Dynamics Analysis

This notebook investigates whether the simulation produces meaningfully diverse behaviour,
or whether it effectively reduces to *agents always going to the same place based on fixed preferences*.

**Four core questions:**
1. **Goal repetition** — How often do agents revisit the same goal node?
2. **Belief dynamics** — Are beliefs (α/β parameters) actually changing across trajectories?
3. **Belief calibration** — Do learned beliefs converge toward the true opening-hours ground truth,
   or do they stay near the uninformative prior?
4. **Belief impact** — How much do beliefs *change* which goal gets selected
   compared to using only preferences?

Data: `run_test_beliefs` (10 agents × 50 trajectories each).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import json
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter, defaultdict
from scipy import stats

from graph_controller.world_graph import WorldGraph
from simulation_controller.belief_store import BeliefStore

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RUN_DIR = "../data/simulation_data/run_test_beliefs"

print("Loading data...")

# --- Trajectories ---
with open(os.path.join(RUN_DIR, "trajectories", "all_trajectories.json")) as f:
    trajectories = json.load(f)   # {agent_id: [traj_dict, ...]}

# --- Agent metadata ---
with open(os.path.join(RUN_DIR, "agents", "all_agents.json")) as f:
    agents_metadata = json.load(f)

# --- Graph ---
G_raw = nx.read_graphml("../data/processed/ucsd_walk_full.graphml")
world_graph = WorldGraph(G_raw)
G = world_graph.G
poi_nodes = world_graph.poi_nodes  # ordered list, length 230

# --- Belief store ---
belief_store = BeliefStore.load(RUN_DIR)

agent_ids = sorted(trajectories.keys())
n_agents  = len(agent_ids)
print(f"  {n_agents} agents, {sum(len(v) for v in trajectories.values())} total trajectories")
print(f"  {len(poi_nodes)} POI nodes")

---
## Analysis 1: Goal Repetition

**Question:** How often do agents visit the same goal node across trajectories?

If the simulation is just preference-driven with no real dynamics, we expect most agents
to repeatedly select the same top-preference node, producing a degenerate dataset.

In [ ]:
# ── Per-agent goal statistics ──────────────────────────────────────────────────

agent_stats = {}

for agent_id in agent_ids:
    trajs = trajectories[agent_id]
    goals = [t['goal_node'] for t in trajs]
    hours = [t['hour']      for t in trajs]
    n = len(goals)

    counter = Counter(goals)
    n_unique = len(counter)

    # Repetition: fraction of trajectories that repeated a previously-seen node
    seen, repeat_count = set(), 0
    for g in goals:
        if g in seen:
            repeat_count += 1
        seen.add(g)
    repetition_rate = repeat_count / n

    # Entropy of goal distribution
    probs = np.array(list(counter.values()), dtype=float) / n
    entropy = float(-np.sum(probs * np.log(probs + 1e-12)))
    max_entropy = np.log(n)   # if each trajectory went to a unique node

    # Most visited node and its fraction
    top_node, top_count = counter.most_common(1)[0]
    top_node_category = G.nodes[top_node].get('Category', 'unknown')

    agent_stats[agent_id] = {
        'n_trajs':        n,
        'n_unique_goals': n_unique,
        'repetition_rate': repetition_rate,
        'goal_entropy':   entropy,
        'max_entropy':    max_entropy,
        'top_node':       top_node,
        'top_node_frac':  top_count / n,
        'top_node_cat':   top_node_category,
        'goal_counter':   counter,
    }

print("Goal Repetition Summary")
print("=" * 60)
print(f"{'Agent':<12} {'N':>4} {'Unique':>7} {'Repeat%':>8} {'Entropy':>8} {'Top node %':>11} {'TopCat':>10}")
print("-" * 60)
for aid, s in agent_stats.items():
    print(f"{aid:<12} {s['n_trajs']:>4} {s['n_unique_goals']:>7} "
          f"{s['repetition_rate']*100:>7.1f}% {s['goal_entropy']:>8.2f} "
          f"{s['top_node_frac']*100:>10.1f}% {s['top_node_cat']:>10}")

rep_rates = [s['repetition_rate'] for s in agent_stats.values()]
top_fracs  = [s['top_node_frac']   for s in agent_stats.values()]
print()
print(f"Mean repetition rate : {np.mean(rep_rates)*100:.1f}%")
print(f"Mean top-node fraction: {np.mean(top_fracs)*100:.1f}% of trajs to the single most-visited node")

In [ ]:
# ── Visualise goal diversity ────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 5, figsize=(18, 7), sharey=False)
axes = axes.flatten()

for idx, (agent_id, s) in enumerate(agent_stats.items()):
    ax = axes[idx]
    counter = s['goal_counter']
    top_k = counter.most_common(15)
    labels = [f"{G.nodes[n].get('Category','?')[:3]}" for n, _ in top_k]
    counts = [c for _, c in top_k]

    colors = sns.color_palette('muted', len(top_k))
    ax.bar(range(len(top_k)), counts, color=colors)
    ax.set_xticks(range(len(top_k)))
    ax.set_xticklabels(labels, rotation=45, fontsize=7)
    ax.set_title(f"{agent_id}\n{s['n_unique_goals']} unique, {s['repetition_rate']*100:.0f}% repeat",
                 fontsize=8)
    ax.set_xlabel("Top 15 goal nodes (category prefix)", fontsize=7)
    ax.set_ylabel("Visit count", fontsize=7)

plt.suptitle("Goal Visit Frequency per Agent (top 15 nodes)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("goal_repetition.png", bbox_inches='tight')
plt.show()
print("Saved goal_repetition.png")

In [ ]:
# ── Population-level: distribution of repetition rates ─────────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(range(n_agents), [s['repetition_rate']*100 for s in agent_stats.values()],
        tick_label=list(agent_stats.keys()))
ax1.axhline(np.mean(rep_rates)*100, color='red', linestyle='--', label=f'Mean = {np.mean(rep_rates)*100:.1f}%')
ax1.set_xlabel("Agent")
ax1.set_ylabel("Repetition rate (%)")
ax1.set_title("Goal Repetition Rate per Agent")
ax1.tick_params(axis='x', rotation=45)
ax1.legend()

ax2.bar(range(n_agents), [s['n_unique_goals'] for s in agent_stats.values()],
        tick_label=list(agent_stats.keys()))
ax2.set_xlabel("Agent")
ax2.set_ylabel("Unique goals visited")
ax2.set_title("Number of Unique Goals per Agent (out of 50 trajs)")
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig("goal_repetition_summary.png", bbox_inches='tight')
plt.show()

---
## Analysis 2: Belief Dynamics

**Question:** Are the agents' beliefs (Beta α/β parameters) actually changing across trajectories,
or are they staying near the initial prior?

We track the α and β parameters at the **start** of each trajectory (cross-trajectory accumulated
knowledge, as stored by `BeliefStore.record_initial_state`). If beliefs are not evolving,
α and β will be close to their initial values (2 or 4) throughout all 50 trajectories.

In [ ]:
# ── Load initial alpha/beta for each trajectory for all agents ─────────────────

N_TRAJS = 50
N_POIS  = len(poi_nodes)

# For each agent: array of shape [N_TRAJS, N_POIS, 24]
agent_alpha = {}  # {agent_id: np.ndarray [N_TRAJS, N_POIS, 24]}
agent_beta  = {}

for agent_id in agent_ids:
    alphas, betas = [], []
    for t in range(N_TRAJS):
        try:
            data = belief_store.get_trajectory(agent_id, t)
        except KeyError:
            break
        alphas.append(data['alpha'])  # [N_POIS, 24] float32
        betas.append(data['beta'])
    agent_alpha[agent_id] = np.stack(alphas, axis=0)  # [n_found, N_POIS, 24]
    agent_beta[agent_id]  = np.stack(betas,  axis=0)

print(f"Loaded alpha/beta: {agent_alpha[agent_ids[0]].shape} per agent (trajs × POIs × hours)")

In [ ]:
# ── How much do α and β grow across trajectories? ────────────────────────────────
# Initial alpha = 2 for most hours, 4 for 9am-5pm
# If a node is visited (within 100m) and observed, alpha or beta gets +1
# We measure: mean(alpha+beta) over all (POI, hour) pairs = precision (total observations)

fig, axes = plt.subplots(2, 5, figsize=(18, 7), sharex=True)
axes = axes.flatten()

for idx, agent_id in enumerate(agent_ids):
    ax = axes[idx]
    alpha_t = agent_alpha[agent_id]  # [N_TRAJS, N_POIS, 24]
    beta_t  = agent_beta[agent_id]

    # Mean precision = mean (alpha + beta) across all POIs and hours per traj
    precision = (alpha_t + beta_t).mean(axis=(1, 2))  # [N_TRAJS]

    # Mean belief (alpha / (alpha+beta)) averaged across all POI-hour pairs
    belief_mean = (alpha_t / (alpha_t + beta_t)).mean(axis=(1, 2))

    # Std of belief across POI-hour pairs (spread = distinguishability)
    belief_std = (alpha_t / (alpha_t + beta_t)).std(axis=(1, 2))

    t = np.arange(len(precision))
    ax.plot(t, precision, label='Mean precision (α+β)', color='steelblue')
    ax.set_ylabel('α + β', color='steelblue', fontsize=8)
    ax.tick_params(axis='y', labelcolor='steelblue')

    ax2 = ax.twinx()
    ax2.plot(t, belief_std, color='darkorange', linestyle='--', label='Belief std')
    ax2.set_ylabel('Belief std', color='darkorange', fontsize=8)
    ax2.tick_params(axis='y', labelcolor='darkorange')

    ax.set_title(f"{agent_id}", fontsize=8)
    ax.set_xlabel("Trajectory #", fontsize=7)

plt.suptitle("Belief Precision (α+β) and Spread Over Trajectories\n"
             "Rising precision = beliefs updated; rising std = beliefs diverging across nodes",
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig("belief_dynamics_precision.png", bbox_inches='tight')
plt.show()

In [ ]:
# ── How many unique POI nodes actually receive observations? ──────────────────────
# A POI gets an observation only if agent passes within 100m of it
# Nodes with alpha+beta > initial prior values have been seen

# Initial prior: alpha=2,beta=2 for hours 0-8,17-23; alpha=4,beta=1 for hours 9-16
# So initial (alpha+beta) for a POI = 24 hours:
#   8 non-business hours × (2+2) + 8 business hours × (4+1) + 8 evening × (2+2)
# But let's just compare traj-0 alpha/beta to traj-N alpha/beta

print("Fraction of POI-hour pairs with alpha or beta > initial prior (observations accumulated)")
print("=" * 75)

# Reconstruct initial prior
init_alpha = np.full((N_POIS, 24), 2.0, dtype=np.float32)
init_alpha[:, 9:17] = 4.0
init_beta  = np.full((N_POIS, 24), 2.0, dtype=np.float32)
init_beta[:, 9:17]  = 1.0

for agent_id in agent_ids:
    alpha_last = agent_alpha[agent_id][-1]  # [N_POIS, 24] at start of last traj
    beta_last  = agent_beta[agent_id][-1]

    # How many POI-hour cells changed?
    alpha_changed = (alpha_last > init_alpha + 0.05)  # +0.05 tolerance (decay noise)
    beta_changed  = (beta_last  > init_beta  + 0.05)
    any_changed   = alpha_changed | beta_changed

    n_changed_cells = any_changed.sum()
    total_cells = N_POIS * 24

    # How many distinct POI nodes had any observation?
    n_pois_seen = any_changed.any(axis=1).sum()

    # Total counts added
    total_alpha_added = (alpha_last - init_alpha).clip(min=0).sum()
    total_beta_added  = (beta_last  - init_beta ).clip(min=0).sum()

    print(f"{agent_id}: POIs seen = {n_pois_seen:>3}/{N_POIS} | "
          f"cells updated = {n_changed_cells:>5}/{total_cells} ({n_changed_cells/total_cells*100:.1f}%) | "
          f"α counts added = {total_alpha_added:.1f}, β counts added = {total_beta_added:.1f}")

In [ ]:
# ── Heatmap: which POI nodes accumulate the most observations? ─────────────────
# Show for one representative agent

EXAMPLE_AGENT = agent_ids[0]
alpha_last = agent_alpha[EXAMPLE_AGENT][-1]   # [N_POIS, 24]
beta_last  = agent_beta[EXAMPLE_AGENT][-1]

# Observations = alpha above prior + beta above prior
obs_count = ((alpha_last - init_alpha).clip(min=0) +
             (beta_last  - init_beta ).clip(min=0))  # [N_POIS, 24]

obs_per_poi = obs_count.sum(axis=1)  # [N_POIS]: total observations per POI

# Get categories for POI nodes
poi_cats = [G.nodes[n].get('Category', '?') for n in poi_nodes]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: top-20 most-observed POI nodes
top20_idx = np.argsort(obs_per_poi)[::-1][:20]
ax1.barh([f"{poi_cats[i][:6]}_{i}" for i in top20_idx],
         obs_per_poi[top20_idx], color='steelblue')
ax1.set_xlabel("Total Bayesian observations accumulated")
ax1.set_title(f"Top 20 Most-Observed POIs\n({EXAMPLE_AGENT}, after 50 trajs)")
ax1.invert_yaxis()

# Right: distribution of observations across all POIs
ax2.hist(obs_per_poi, bins=30, color='steelblue', edgecolor='white')
ax2.axvline(0.05, color='red', linestyle='--', label='threshold (>0.05 = observed)')
n_zero = (obs_per_poi < 0.05).sum()
ax2.set_xlabel("Total observations per POI")
ax2.set_ylabel("Number of POIs")
ax2.set_title(f"Distribution of Observations Per POI\n({n_zero}/{N_POIS} POIs never observed)")
ax2.legend()

plt.tight_layout()
plt.savefig("belief_observation_coverage.png", bbox_inches='tight')
plt.show()

---
## Analysis 3: Belief Calibration vs. Ground Truth

**Question:** Do the agents' learned beliefs accurately reflect the true opening hours?

For each POI node at each hour, we know the ground truth: open (1) or closed (0).
We compare the agent's `temporal_belief` (α/(α+β)) to this ground truth.

If beliefs are well-calibrated:
- Nodes that are open at hour *h* should have high `temporal_belief[h]`
- Nodes that are closed at hour *h* should have low `temporal_belief[h]`

If beliefs are stuck near the prior, all beliefs ≈ 0.5–0.8 (business-hours prior),
regardless of ground truth.

In [ ]:
# ── Build ground truth open/closed matrix [N_POIS, 24] ────────────────────────

def is_open_at_hour(node_id, hour, G):
    node_data = G.nodes[node_id]
    hours_info = node_data.get('opening_hours')
    if not hours_info or not isinstance(hours_info, dict):
        return None  # No info
    open_h  = hours_info.get('open', 0)
    close_h = hours_info.get('close', 24)
    if close_h < open_h:  # wraparound
        return hour >= open_h or hour < close_h
    return open_h <= hour < close_h

gt_open = np.full((N_POIS, 24), np.nan)  # NaN = no opening hours info
for i, node_id in enumerate(poi_nodes):
    for h in range(24):
        result = is_open_at_hour(node_id, h, G)
        if result is not None:
            gt_open[i, h] = float(result)

n_with_hours = np.isfinite(gt_open).any(axis=1).sum()
print(f"POI nodes with opening_hours info: {n_with_hours} / {N_POIS}")
print(f"POI nodes without opening_hours:   {N_POIS - n_with_hours} (assumed always open)")

# Among nodes with hours, what fraction of hours are 'open'?
mask = np.isfinite(gt_open)
frac_open = gt_open[mask].mean()
print(f"Fraction of (node, hour) pairs that are OPEN: {frac_open*100:.1f}%")

In [ ]:
# ── Compute final beliefs for each agent and compare to ground truth ───────────

# Final belief = alpha / (alpha + beta) at the start of the last trajectory
# (accumulated over first 49 trajectories)

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

corr_results = {}

for idx, agent_id in enumerate(agent_ids):
    ax = axes[idx]
    alpha_final = agent_alpha[agent_id][-1]  # [N_POIS, 24]
    beta_final  = agent_beta[agent_id][-1]
    belief_final = alpha_final / (alpha_final + beta_final)  # [N_POIS, 24]

    # Only look at POI-hour pairs where we have ground truth
    valid = np.isfinite(gt_open)  # [N_POIS, 24] bool

    gt_vals    = gt_open[valid].astype(float)
    pred_vals  = belief_final[valid].astype(float)

    # Prior belief for comparison
    prior_beliefs = init_alpha / (init_alpha + init_beta)  # [N_POIS, 24]
    prior_vals    = prior_beliefs[valid]

    # Pearson correlation
    r_belief, p_belief = stats.pearsonr(gt_vals, pred_vals)
    r_prior,  p_prior  = stats.pearsonr(gt_vals, prior_vals)

    # MAE vs ground truth
    mae_belief = np.abs(pred_vals - gt_vals).mean()
    mae_prior  = np.abs(prior_vals - gt_vals).mean()

    corr_results[agent_id] = {
        'r_belief': r_belief, 'r_prior': r_prior,
        'mae_belief': mae_belief, 'mae_prior': mae_prior,
    }

    # Scatter: ground truth vs final belief
    ax.scatter(gt_vals + np.random.normal(0, 0.02, len(gt_vals)),
               pred_vals, alpha=0.05, s=3, color='steelblue', label=f'r={r_belief:.2f}')
    ax.axhline(0.5, color='gray', linestyle=':', linewidth=0.8)
    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('Ground truth (open=1, closed=0)', fontsize=7)
    ax.set_ylabel('Agent belief P(open)', fontsize=7)
    ax.set_title(f"{agent_id}\nr={r_belief:.2f}, MAE={mae_belief:.3f}", fontsize=8)
    ax.legend(fontsize=7)

plt.suptitle("Belief Calibration: Agent Beliefs vs. Ground Truth Opening Hours\n"
             "(Jitter added to 0/1 ground truth; r close to 0 = no learning)",
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig("belief_calibration_scatter.png", bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary table: how much did beliefs improve over the prior? ─────────────────

print("Belief Calibration Summary (vs ground truth opening hours)")
print("=" * 75)
print(f"{'Agent':<12} {'r(belief,GT)':>13} {'r(prior,GT)':>12} {'MAE_belief':>11} {'MAE_prior':>10} {'MAE_improvement':>16}")
print("-" * 75)
for aid, s in corr_results.items():
    improvement = s['mae_prior'] - s['mae_belief']
    print(f"{aid:<12} {s['r_belief']:>13.3f} {s['r_prior']:>12.3f} "
          f"{s['mae_belief']:>11.4f} {s['mae_prior']:>10.4f} {improvement:>+15.4f}")

print()
print("NOTE: r(prior, GT) tells us how much the initialization already encodes the ground truth.")
print("If r(belief,GT) ≈ r(prior,GT), agents are NOT learning anything beyond the initial prior.")

In [ ]:
# ── Per-hour analysis: do beliefs at specific hours improve? ───────────────────
# Focus on hours 0-5 (late night, typically closed) and 9-17 (open hours)
# The prior already has business-hours bias, so we should check if beliefs
# diverge further in the correct direction.

hours_of_interest = [0, 2, 6, 9, 12, 14, 20, 22]  # late night, morning, peak, evening

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

# Use mean across all agents for final beliefs
mean_belief_final = np.stack(
    [agent_alpha[aid][-1] / (agent_alpha[aid][-1] + agent_beta[aid][-1])
     for aid in agent_ids], axis=0
).mean(axis=0)  # [N_POIS, 24]

for ax, h in zip(axes, hours_of_interest):
    gt_h = gt_open[:, h]  # [N_POIS]
    belief_h = mean_belief_final[:, h]
    prior_h  = init_alpha[:, h] / (init_alpha[:, h] + init_beta[:, h])

    # Only nodes with ground truth
    valid_h = np.isfinite(gt_h)
    if valid_h.sum() == 0:
        ax.set_title(f"Hour {h:02d}:00 — no GT data")
        continue

    gt_v     = gt_h[valid_h]
    belief_v = belief_h[valid_h]
    prior_v  = prior_h[valid_h]

    # Group by open/closed
    open_mask   = gt_v == 1
    closed_mask = gt_v == 0

    bp_data = []
    bp_labels = []
    if open_mask.sum() > 0:
        bp_data.extend([prior_v[open_mask], belief_v[open_mask]])
        bp_labels.extend([f'Prior\n(open n={open_mask.sum()})', 'Belief\n(open)'])
    if closed_mask.sum() > 0:
        bp_data.extend([prior_v[closed_mask], belief_v[closed_mask]])
        bp_labels.extend([f'Prior\n(closed n={closed_mask.sum()})', 'Belief\n(closed)'])

    ax.boxplot(bp_data, labels=bp_labels, patch_artist=True,
               boxprops=dict(facecolor='lightblue', color='steelblue'))
    ax.axhline(0.5, color='red', linestyle='--', linewidth=0.8, label='chance')
    ax.set_ylim(0, 1)
    ax.set_title(f"Hour {h:02d}:00", fontsize=9)
    ax.set_ylabel("P(open) belief", fontsize=8)
    ax.tick_params(axis='x', labelsize=7)

plt.suptitle("Prior vs. Learned Belief by Hour and True Status\n"
             "Well-calibrated: open nodes have high belief, closed nodes have low belief",
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig("belief_calibration_by_hour.png", bbox_inches='tight')
plt.show()

---
## Analysis 4: Counterfactual — Do Beliefs Actually Change Goal Selection?

**Question:** If we replaced each agent's beliefs with the flat prior (uniform 0.5 or
initial prior beliefs), would the same goals get selected?

Goal selection weight = `preference(node) × belief(node, hour)`.  
If beliefs are nearly identical across all candidate nodes at the sampled hour,
the belief term cancels out and selection reduces to **pure preference sampling**.

We measure:
- **Rank correlation** between `pref-only` and `pref × belief` probability distributions
- **Top-1 agreement**: does the top node change when beliefs are included vs. excluded?
- **Kullback-Leibler divergence** between the two distributions
- **Belief variance across candidate nodes** at each sampled hour

In [ ]:
# ── For each trajectory: reconstruct the goal selection distribution ───────────
#
# We cannot re-run the exact decision (we don't have the exact α/β at decision time),
# but we have α/β at the START of the trajectory.
# We compute, for each category chosen in the trajectory:
#   (a) P(node | pref only)
#   (b) P(node | pref × belief)
# and compare them.

CAT_ATTRS = ["home", "study", "food", "leisure", "errands", "health"]

# Build a lookup: node_id -> category
node_to_cat = {n: G.nodes[n].get('Category') for n in poi_nodes}

results = []  # list of dicts, one per (agent, traj)

for agent_id in agent_ids:
    trajs = trajectories[agent_id]
    agent_meta = agents_metadata[agent_id]

    for t_idx, traj in enumerate(trajs):
        hour       = traj['hour']
        goal_node  = traj['goal_node']
        goal_cat   = node_to_cat.get(goal_node)

        if goal_cat not in CAT_ATTRS:
            continue  # agent went home after failed attempts

        # Retrieve α/β at start of this trajectory
        try:
            bdata = belief_store.get_trajectory(agent_id, t_idx)
        except KeyError:
            continue
        alpha_t = bdata['alpha'].astype(np.float64)  # [N_POIS, 24]
        beta_t  = bdata['beta'].astype(np.float64)

        # Compute temporal_belief at the sampled hour for all POIs
        belief_at_hour = alpha_t[:, hour] / (alpha_t[:, hour] + beta_t[:, hour])  # [N_POIS]

        # Get node preferences for the chosen category
        cat_prefs = agent_meta.get(f"{goal_cat}_preferences", {})
        if not cat_prefs:
            continue

        # Build arrays for candidate nodes in this category
        cand_nodes = list(cat_prefs.keys())
        pref_arr   = np.array([cat_prefs[n] for n in cand_nodes], dtype=np.float64)

        # Map candidate nodes to POI index
        poi_idx_map = {n: belief_store.poi_to_idx.get(n) for n in cand_nodes}
        belief_arr  = np.array([
            belief_at_hour[poi_idx_map[n]] if poi_idx_map[n] is not None else 0.5
            for n in cand_nodes
        ], dtype=np.float64)

        # (a) Preference-only distribution
        pref_dist = pref_arr / pref_arr.sum()

        # (b) Preference × belief distribution
        combined  = pref_arr * belief_arr
        total     = combined.sum()
        if total <= 0:
            continue
        combined_dist = combined / total

        # Rank correlation (Spearman) between distributions
        rho, _ = stats.spearmanr(pref_dist, combined_dist)

        # Top-1 agreement
        top_pref    = cand_nodes[np.argmax(pref_dist)]
        top_combined = cand_nodes[np.argmax(combined_dist)]
        top1_agree  = int(top_pref == top_combined)

        # KL(combined || pref_only)
        eps = 1e-12
        kl = float(np.sum(combined_dist * np.log((combined_dist + eps) / (pref_dist + eps))))

        # Belief variance across candidate nodes at this hour
        belief_var = float(np.var(belief_arr))
        belief_range = float(np.max(belief_arr) - np.min(belief_arr))

        results.append({
            'agent_id':     agent_id,
            'traj_idx':     t_idx,
            'hour':         hour,
            'category':     goal_cat,
            'n_candidates': len(cand_nodes),
            'spearman_rho': rho,
            'top1_agree':   top1_agree,
            'kl_div':       kl,
            'belief_var':   belief_var,
            'belief_range': belief_range,
        })

print(f"Computed counterfactual analysis for {len(results)} trajectories")

In [ ]:
import numpy as np

# ── Aggregate summary ──────────────────────────────────────────────────────────

rhos        = np.array([r['spearman_rho'] for r in results])
top1_agrees = np.array([r['top1_agree']   for r in results])
kl_divs     = np.array([r['kl_div']       for r in results])
belief_vars = np.array([r['belief_var']   for r in results])
bel_ranges  = np.array([r['belief_range'] for r in results])

print("=" * 65)
print("COUNTERFACTUAL ANALYSIS: Do Beliefs Change Goal Selection?")
print("=" * 65)
print(f"  Spearman ρ (pref-only vs pref×belief):")
print(f"    Mean = {rhos.mean():.4f}  (1.0 = identical ranking)")
print(f"    Std  = {rhos.std():.4f}")
print(f"    Min  = {rhos.min():.4f}")
print()
print(f"  Top-1 node agreement (pref-only vs pref×belief):")
print(f"    {top1_agrees.mean()*100:.1f}% of trajectories select the SAME top node")
print()
print(f"  KL divergence (combined || pref-only):")
print(f"    Mean = {kl_divs.mean():.6f} nats  (0 = identical distributions)")
print(f"    Max  = {kl_divs.max():.6f}")
print()
print(f"  Belief variance across candidate nodes at sampled hour:")
print(f"    Mean = {belief_vars.mean():.6f}  (low = beliefs nearly uniform)")
print(f"    Mean range = {bel_ranges.mean():.4f}  (max - min belief across candidates)")
print()
print("INTERPRETATION:")
if rhos.mean() > 0.99:
    print("  ⚠️  Beliefs have essentially NO impact on goal ranking — near-identical to pref-only.")
elif rhos.mean() > 0.95:
    print("  ⚠️  Beliefs have very minimal impact on goal ranking.")
elif rhos.mean() > 0.80:
    print("  ⚡ Beliefs have moderate impact — rankings shift somewhat.")
else:
    print("  ✅ Beliefs meaningfully alter goal selection.")

In [ ]:
# ── Visualise: belief impact by category and hour ─────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# KL divergence by category
cats_present = sorted(set(r['category'] for r in results))
kl_by_cat = {c: np.array([r['kl_div'] for r in results if r['category'] == c])
             for c in cats_present}
axes[0].boxplot([kl_by_cat[c] for c in cats_present], labels=cats_present, patch_artist=True,
                boxprops=dict(facecolor='lightcoral', color='darkred'))
axes[0].set_title("KL divergence by Goal Category")
axes[0].set_ylabel("KL(pref×belief ∥ pref-only) [nats]")
axes[0].set_xlabel("Category")
axes[0].tick_params(axis='x', rotation=30)

# Belief range vs KL divergence scatter
axes[1].scatter(bel_ranges, kl_divs, alpha=0.3, s=15, color='steelblue')
axes[1].set_xlabel("Belief range (max − min across candidates)")
axes[1].set_ylabel("KL divergence")
axes[1].set_title("Does Belief Spread Drive Distributional Shift?")
r_val, _ = stats.pearsonr(bel_ranges, kl_divs)
axes[1].text(0.05, 0.92, f"r = {r_val:.3f}", transform=axes[1].transAxes, fontsize=10)

# Top-1 agreement by hour
hours_arr = np.array([r['hour'] for r in results])
agree_arr = np.array([r['top1_agree'] for r in results])
hourly_agree = {h: agree_arr[hours_arr == h].mean() for h in range(24)
                if (hours_arr == h).sum() > 0}
hs = sorted(hourly_agree.keys())
axes[2].bar(hs, [hourly_agree[h] for h in hs], color='steelblue')
axes[2].axhline(top1_agrees.mean(), color='red', linestyle='--',
                label=f'Overall mean = {top1_agrees.mean()*100:.1f}%')
axes[2].set_xlabel("Hour of day")
axes[2].set_ylabel("Fraction where pref-only top-1 == pref×belief top-1")
axes[2].set_title("Top-1 Goal Agreement by Hour")
axes[2].legend(fontsize=8)

plt.suptitle("Belief Impact on Goal Selection Distributions", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("belief_impact_counterfactual.png", bbox_inches='tight')
plt.show()

In [ ]:
# ── Deep dive: distribution of belief values at each simulated hour ────────────
# If all beliefs are clustered tightly, the belief term is nearly constant → no impact

fig, ax = plt.subplots(figsize=(14, 5))

# Collect all belief values at each hour for the first agent's last trajectory
EXAMPLE_AGENT = agent_ids[0]
alpha_final = agent_alpha[EXAMPLE_AGENT][-1]  # [N_POIS, 24]
beta_final  = agent_beta[EXAMPLE_AGENT][-1]
belief_matrix = alpha_final / (alpha_final + beta_final)  # [N_POIS, 24]

# Violin plot of belief values across POIs for each hour
belief_list = [belief_matrix[:, h] for h in range(24)]

parts = ax.violinplot(belief_list, positions=range(24), showmedians=True)
for pc in parts['bodies']:
    pc.set_facecolor('steelblue')
    pc.set_alpha(0.5)

# Overlay prior
prior_means = init_alpha[:, :].mean(axis=0) / (init_alpha[:, :].mean(axis=0) + init_beta[:, :].mean(axis=0))
ax.plot(range(24), prior_means, 'r--', linewidth=2, label='Prior mean')

ax.set_xlabel("Hour of day")
ax.set_ylabel("P(open) belief across all POI nodes")
ax.set_title(f"Distribution of Beliefs Across POIs at Each Hour\n"
             f"({EXAMPLE_AGENT}, after 50 trajs) — Narrow violin = all POIs have similar beliefs")
ax.set_xticks(range(24))
ax.set_xticklabels([f"{h:02d}" for h in range(24)], fontsize=8)
ax.legend()

plt.tight_layout()
plt.savefig("belief_spread_by_hour.png", bbox_inches='tight')
plt.show()

---
## Analysis 5: Belief Update Opportunity — Spatial Coverage

**Question:** Can agents even *encounter* most POI nodes within 100m during traversal?

Belief updates only happen when passing within 100m of a POI.
If trajectories mostly pass through the same small set of corridors,
beliefs for the majority of POIs never update — they just slowly decay back to prior.

We check: how many distinct nodes does each trajectory pass through,
and how many POIs fall within 100m of any traversed node?

In [ ]:
import haversine as hs

# ── Precompute: for each graph node, which POIs are within 100m? ───────────────
# This is expensive — we'll sample a subset of traversal nodes instead.

# For each agent and trajectory, count unique path nodes and estimate POI coverage
print("Path length and spatial coverage per agent (mean across trajectories)")
print("=" * 70)

BELIEF_DIST = 100  # meters, from simulation_runner

for agent_id in agent_ids:
    trajs = trajectories[agent_id]
    path_lens = []

    for traj in trajs:
        path = traj['path']  # list of [node_id, goal_node] (JSON array)
        path_nodes = [p[0] for p in path]
        path_lens.append(len(path_nodes))

    print(f"{agent_id}: mean path length = {np.mean(path_lens):.1f} nodes "
          f"(min={min(path_lens)}, max={max(path_lens)})")

print()
print(f"Belief update check: updates every 5 nodes along path")
print(f"Belief update radius: {BELIEF_DIST}m")
print(f"→ Only POIs within {BELIEF_DIST}m of a check-point node get updated")

In [ ]:
# ── Sample: for one agent, track which POIs are updated across all trajectories ─

EXAMPLE_AGENT = agent_ids[0]
trajs = trajectories[EXAMPLE_AGENT]

# Precompute POI coordinates
poi_coords = np.array([[G.nodes[n]['y'], G.nodes[n]['x']] for n in poi_nodes])  # [N_POIS, 2] (lat, lon)

poi_seen_per_traj = []
for traj in trajs:
    path_nodes = [p[0] for p in traj['path']]
    # Check every 5th node (matching simulation)
    check_nodes = path_nodes[::5]

    pois_seen_this_traj = set()
    for node_id in check_nodes:
        if not G.has_node(node_id):
            continue
        node_lat = G.nodes[node_id]['y']
        node_lon = G.nodes[node_id]['x']

        dists = np.array([
            hs.haversine((node_lat, node_lon), (plat, plon), unit=hs.Unit.METERS)
            for plat, plon in poi_coords
        ])  # [N_POIS]

        nearby = np.where(dists <= BELIEF_DIST)[0]
        for i in nearby:
            pois_seen_this_traj.add(poi_nodes[i])

    poi_seen_per_traj.append(pois_seen_this_traj)

# Cumulative unique POIs seen after each trajectory
cum_seen = set()
cum_counts = []
for seen in poi_seen_per_traj:
    cum_seen |= seen
    cum_counts.append(len(cum_seen))

print(f"Agent {EXAMPLE_AGENT}: unique POIs encountered within {BELIEF_DIST}m")
for t in [1, 5, 10, 25, 50]:
    idx = min(t, len(cum_counts)) - 1
    print(f"  After {t:>2} trajs: {cum_counts[idx]:>3} / {N_POIS} POIs "
          f"({cum_counts[idx]/N_POIS*100:.1f}%)")

In [ ]:
# ── Plot cumulative POI coverage ───────────────────────────────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: cumulative coverage curve
ax1.plot(range(1, len(cum_counts)+1), cum_counts, marker='o', markersize=3, color='steelblue')
ax1.axhline(N_POIS, color='red', linestyle='--', label=f'All {N_POIS} POIs')
ax1.set_xlabel("Trajectory number")
ax1.set_ylabel("Cumulative unique POIs encountered")
ax1.set_title(f"POI Coverage Over Trajectories\n({EXAMPLE_AGENT})")
ax1.legend()

# Right: per-trajectory POI encounter count
per_traj_counts = [len(s) for s in poi_seen_per_traj]
ax2.bar(range(len(per_traj_counts)), per_traj_counts, color='steelblue', alpha=0.7)
ax2.axhline(np.mean(per_traj_counts), color='red', linestyle='--',
            label=f'Mean = {np.mean(per_traj_counts):.1f}')
ax2.set_xlabel("Trajectory #")
ax2.set_ylabel("POIs encountered within 100m")
ax2.set_title(f"POIs Observed Per Trajectory\n({EXAMPLE_AGENT})")
ax2.legend()

plt.suptitle("Spatial Coverage of Belief Updates", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("belief_spatial_coverage.png", bbox_inches='tight')
plt.show()

---
## Analysis 6: Cross-Agent Goal Concentration

**Question:** Are too many agents going to the exact same nodes?

If 43% of trips end at the same node, the dataset lacks diversity and the model
might just learn to always predict that popular node.

In [ ]:
# ── Cross-agent goal concentration ─────────────────────────────────────────────

all_goals = []
for agent_id in agent_ids:
    for traj in trajectories[agent_id]:
        all_goals.append(traj['goal_node'])

global_counter = Counter(all_goals)
total_trips = len(all_goals)

print(f"Total trajectory goals: {total_trips}")
print(f"Unique goal nodes:       {len(global_counter)}")
print()
print("Top 20 most-visited goal nodes (all agents combined):")
print(f"{'Rank':<5} {'Node (truncated)':<35} {'Category':<12} {'Visits':>7} {'% of all':>10}")
print("-" * 70)
for rank, (node, count) in enumerate(global_counter.most_common(20), 1):
    cat = G.nodes[node].get('Category', 'unknown')
    name = G.nodes[node].get('name', node[:25])
    print(f"{rank:<5} {str(name)[:34]:<35} {cat:<12} {count:>7} {count/total_trips*100:>9.1f}%")

print()
# Top-1 node concentration
top1_count = global_counter.most_common(1)[0][1]
print(f"Top-1 node captures {top1_count/total_trips*100:.1f}% of all trips")

# Top-5 cumulative
top5_sum = sum(c for _, c in global_counter.most_common(5))
print(f"Top-5 nodes capture {top5_sum/total_trips*100:.1f}% of all trips")

# Distribution: how many nodes capture 80% of trips?
sorted_counts = sorted(global_counter.values(), reverse=True)
cumsum = np.cumsum(sorted_counts)
nodes_for_80pct = int(np.searchsorted(cumsum, 0.8 * total_trips)) + 1
print(f"Nodes needed to cover 80% of trips: {nodes_for_80pct} out of {len(global_counter)}")

In [ ]:
# ── Lorenz curve of goal concentration ─────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Lorenz curve
sorted_counts_asc = np.array(sorted(global_counter.values()))
cumsum_norm = np.cumsum(sorted_counts_asc) / sorted_counts_asc.sum()
x = np.linspace(0, 1, len(sorted_counts_asc))
axes[0].plot(x, cumsum_norm, color='steelblue', label='Goal distribution')
axes[0].plot([0, 1], [0, 1], 'r--', label='Perfect equality (uniform)')
axes[0].fill_between(x, x, cumsum_norm, alpha=0.2, color='steelblue')
gini = 1 - 2 * np.trapz(cumsum_norm, x)
axes[0].set_title(f"Lorenz Curve of Goal Visits\nGini = {gini:.3f} (0=uniform, 1=all to one node)")
axes[0].set_xlabel("Fraction of goal nodes (sorted by visit count)")
axes[0].set_ylabel("Fraction of total visits")
axes[0].legend()

# Top 30 nodes bar chart
top30 = global_counter.most_common(30)
labels = [G.nodes[n].get('Category', '?')[:3] + f'_{i}' for i, (n, _) in enumerate(top30)]
counts_30 = [c for _, c in top30]
colors = sns.color_palette('muted', 30)
axes[1].bar(range(30), [c/total_trips*100 for c in counts_30], color=colors)
axes[1].set_xticks(range(30))
axes[1].set_xticklabels(labels, rotation=45, fontsize=7)
axes[1].set_xlabel("Goal node (category prefix)")
axes[1].set_ylabel("% of all trips")
axes[1].set_title("Top 30 Goal Nodes (All Agents Combined)")

plt.suptitle("Cross-Agent Goal Concentration", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("goal_concentration.png", bbox_inches='tight')
plt.show()

---
## Summary and Diagnosis

Collects all findings into a concise diagnosis.

In [ ]:
print("="*70)
print("SIMULATION DYNAMICS — DIAGNOSTIC SUMMARY")
print("="*70)

print()
print("[1] GOAL REPETITION")
print(f"    Mean repetition rate:       {np.mean(rep_rates)*100:.1f}% (fraction of trajs revisiting a seen node)")
print(f"    Mean unique goals per agent: {np.mean([s['n_unique_goals'] for s in agent_stats.values()]):.1f} / 50 trajectories")
print(f"    Mean top-node concentration: {np.mean(top_fracs)*100:.1f}% of trips to the single most-visited node")
print()

print("[2] BELIEF DYNAMICS")
all_alpha_changes = []
all_beta_changes  = []
for aid in agent_ids:
    a0 = agent_alpha[aid][0]  # traj 0 start
    aN = agent_alpha[aid][-1] # traj N start
    all_alpha_changes.append((aN - init_alpha).clip(min=0).sum())
    b0 = agent_beta[aid][0]
    bN = agent_beta[aid][-1]
    all_beta_changes.append((bN - init_beta).clip(min=0).sum())
print(f"    Mean total α increments accumulated: {np.mean(all_alpha_changes):.1f}")
print(f"    Mean total β increments accumulated: {np.mean(all_beta_changes):.1f}")
print(f"    (Total possible ≈ 50 trajs × avg observations per traj)")
print()

print("[3] BELIEF CALIBRATION vs. GROUND TRUTH")
mean_r_belief = np.mean([s['r_belief'] for s in corr_results.values()])
mean_r_prior  = np.mean([s['r_prior']  for s in corr_results.values()])
mean_mae_b = np.mean([s['mae_belief'] for s in corr_results.values()])
mean_mae_p = np.mean([s['mae_prior']  for s in corr_results.values()])
print(f"    r(belief, ground_truth) = {mean_r_belief:.3f}  vs  r(prior, ground_truth) = {mean_r_prior:.3f}")
print(f"    MAE(belief) = {mean_mae_b:.4f}  vs  MAE(prior) = {mean_mae_p:.4f}")
print(f"    Improvement over prior = {(mean_mae_p - mean_mae_b):.4f}")
print()

print("[4] BELIEF IMPACT ON GOAL SELECTION")
print(f"    Mean Spearman ρ (pref-only vs pref×belief): {rhos.mean():.4f}")
print(f"    Top-1 agreement: {top1_agrees.mean()*100:.1f}%")
print(f"    Mean KL divergence: {kl_divs.mean():.6f} nats")
print(f"    Mean belief range across candidates: {bel_ranges.mean():.4f}")
print()

print("[5] CROSS-AGENT GOAL CONCENTRATION")
print(f"    Top-1 node: {top1_count/total_trips*100:.1f}% of all trips")
print(f"    Top-5 nodes: {top5_sum/total_trips*100:.1f}% of all trips")
print(f"    Gini coefficient: {gini:.3f}")
print(f"    Nodes for 80% coverage: {nodes_for_80pct} of {len(global_counter)} visited")
print()

print("[6] SPATIAL COVERAGE FOR BELIEF UPDATES")
print(f"    After 50 trajs, agent sees {cum_counts[-1]}/{N_POIS} POIs within 100m ({cum_counts[-1]/N_POIS*100:.1f}%)")
print(f"    Mean POIs encountered per trajectory: {np.mean(per_traj_counts):.1f}")